<a href="https://colab.research.google.com/github/Fespinoza1477/2026-10-09-Public-API-to-Clean-Table-Lab.ipynb/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time
from datetime import datetime

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last_err = None
    for attempt in range(tries):
      try:
        r = requests.get(BASE, params=params, timeout=15)
        if r.status_code == 200:
          return pd.DataFrame(r.json()['daily'])

        if r.status_code not in RETRY_STATUSES:
          raise RuntimeError(f'Not retryable: {r.status_code} {r.text[:200]}')
        last_err = f'Status {r.status_code}'
      except requests.RequestException as e:
        last_err = repr(e)
      if attempt < tries - 1:
        delay = 0.5 * (2 ** attempt)
        print(f'Attempt {attempt + 1} failed ({last_err}); retrying in {delay}s')
        time.sleep(delay)
    raise RuntimeError(f'Gave up after {tries} attempts: {last_err}')
    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})
used_fallback = False
try:
  raw_wx = get_weather('2026-09-05', '2026-10-03')
  wx = raw_wx[raw_wx['time'].isin(['2026-09-05', '2026-10-03'])].copy()
  wx['source'] = 'open-meteo'
except Exception as e:
  print(f"Network failed: {e}. Utilizing fallback sample.")
  used_fallback = True
  wx = FALLBACK.copy()
  wx['source'] = 'fallback sample'

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
  wx = FALLBACK.copy()
  wx['source'] = 'fallback sample'

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns={
    'time' : 'date',
    'temperature_2m_max' : 'temp_max',
    'precipitation_sum' : 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
wx_sorted = wx.sort_values('precip_mm', ascending=False)
wetter_day = wx_sorted.iloc[0]
drier_day = wx_sorted.iloc[1]
diff = wetter_day['precip_mm'] - drier_day['precip_mm']

print(f" The weekend of {wetter_day['date'].strftime('%Y-%m-%d')} was wetter by {diff:.1f} mm.")

 The weekend of 2026-10-03 was wetter by 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
print('Source:     Open_meteo historical archive API')
print('Endpoint: ', BASE)
print('Coords:  38.03, -78.51')
print('Dates:   2026-09-05 and 2026-10-03')
print('Pulled:   ', datetime.now().strftime('%Y-%m-%d %H:%M'))
print('Types:   ', 'Fallback sample' if used_fallback else 'Real data')

Source:     Open_meteo historical archive API
Endpoint:  https://archive-api.open-meteo.com/v1/archive
Coords:  38.03, -78.51
Dates:   2026-09-05 and 2026-10-03
Pulled:    2026-10-08 00:47
Types:    Real data


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If a failing function returns an empty DataFrame instead of raising an error, it behaves as a silent failure disguised as data. When you perform an inner join between 200 days of sales records and this empty weather table, the join will produce an empty DataFrame, silently erasing all of your sales data. If you use a left join instead, the pipeline will pad all 200 days with null weather values. On a chart, this null precipitation could be falsely interpreted as a historic, 200-day drought occuring alongside normal revenue, leading you to present a network outage as actual business intelligence.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
try:
  orl_raw = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
  orl = orl_raw[orl_raw['time'].isin(['2026-09-05', '2026-10-03'])].copy()
  orl = orl.rename(columns={
      'time': 'date',
      'temperature_2m_max': 'temp_max',
      'precipitation_sum': 'precip_mm'
  })
  orl['date'] = pd.to_datetime(orl['date'])
  orl['source'] = 'open-meteo'
  orl['city'] = 'Orlando'

  cville = wx.copy()
  cville['city'] = 'Charlottesville'
  stretch_df = pd.concat([cville, orl], ignore_index=True)
  stretch_df = stretch_df[['date', 'city', 'temp_max', 'precip_mm', 'source']]
  print("\n--- Stretch Output ---")
  print(stretch_df)
except Exception as e:
  print("Stretch goal failed to execute:", e)


--- Stretch Output ---
        date             city  temp_max  precip_mm      source
0 2026-09-05  Charlottesville      29.4        2.5  open-meteo
1 2026-10-03  Charlottesville      22.0       14.7  open-meteo
2 2026-09-05          Orlando      32.1        3.9  open-meteo
3 2026-10-03          Orlando      30.4       14.5  open-meteo
